# Section 2 — Giving Agents Tools (Local Ollama)

A **new local tools exercise** with fictional inventory, a deterministic allowlisted tool dispatcher and Ollama-based explanation. This is **not** the original Strands @tool or AWS S3/DynamoDB/Lambda lab. Run Code Cells 1–8 sequentially; all data fictional.

## Cell 1 — Ollama availability

In [ ]:
import json, urllib.request, time
from pathlib import Path

BASE = "http://127.0.0.1:11434"
MODEL = "qwen3:1.7b"
with urllib.request.urlopen(BASE + "/api/tags", timeout=10) as response:
    model_names = [x.get("name") for x in json.load(response).get("models", [])]
assert MODEL in model_names, "Run: ollama pull qwen3:1.7b"
print("PASS: Ollama available:", MODEL)

## Cell 2 — Define deterministic Python tools

In [ ]:
INVENTORY = {"SKU-101": 12, "SKU-102": 0, "SKU-103": 7}

def calculate_tip(bill, percent):
    if not isinstance(bill, (int, float)) or bill < 0:
        raise ValueError("Bill must be a positive amount")
    if not isinstance(percent, (int, float)) or not 0 <= percent <= 100:
        raise ValueError("Tip percent must be between 0 and 100")
    return {"tip": round(bill * percent / 100, 2), "total": round(bill * (1 + percent / 100), 2)}

def check_inventory(sku):
    if not isinstance(sku, str):
        raise ValueError("SKU must be a string")
    if sku not in INVENTORY:
        return {"sku": sku, "exists": False, "quantity": 0}
    return {"sku": sku, "exists": True, "quantity": INVENTORY[sku]}

print("PASS: Tip calculator and fictional inventory tool defined")

## Cell 3 — Unit-test both tools

In [ ]:
assert calculate_tip(200, 10) == {"tip": 20.0, "total": 220.0}
assert check_inventory("SKU-101")["quantity"] == 12
assert check_inventory("SKU-102")["quantity"] == 0
assert check_inventory("MISSING")["exists"] is False
print("PASS: Both tools return grounded deterministic values")

## Cell 4 — Allowlisted tool dispatcher

In [ ]:
ALLOWED_TOOLS = {"calculate_tip": calculate_tip, "check_inventory": check_inventory}
TRACE = []

def dispatch_tool(tool_name, kwargs):
    if tool_name not in ALLOWED_TOOLS:
        raise PermissionError("Tool not on the allowlist")
    started = time.monotonic()
    outcome = ALLOWED_TOOLS[tool_name](**kwargs)
    TRACE.append({
        "tool": tool_name,
        "args": dict(kwargs),
        "result": outcome,
        "duration_s": round(time.monotonic() - started, 4),
    })
    return outcome

assert dispatch_tool("calculate_tip", {"bill": 500, "percent": 12})["total"] == 560.0
print("PASS: Dispatcher and tool-use trace ready")

## Cell 5 — Run a structured fictional task

In [ ]:
stock_result = dispatch_tool("check_inventory", {"sku": "SKU-103"})
tip_result = dispatch_tool("calculate_tip", {"bill": 250, "percent": 10})
assert stock_result["quantity"] == 7
assert tip_result["total"] == 275.0
assert len(TRACE) == 3
print("Stock:", stock_result, "Tip:", tip_result)
print("PASS: Tools called with explicit parameters")

## Cell 6 — Ask local Qwen3 to explain tool results

In [ ]:
def explain_tool_results(data):
    payload = {
        "model": MODEL, "stream": False, "think": False,
        "messages": [
            {"role": "system", "content": "Explain exact structured tool results. Never invent stock or amounts."},
            {"role": "user", "content": "Explain in 2 sentences this fictional tool data: " + json.dumps(data)}
        ],
        "options": {"temperature": 0, "num_predict": 500}
    }
    for attempt in range(2):
        req = urllib.request.Request(BASE + "/api/chat",
            data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json"}, method="POST")
        with urllib.request.urlopen(req, timeout=210) as response:
            message = json.load(response).get("message") or {}
        answer = str(message.get("content") or "").strip()
        if answer:
            return answer
        payload["options"]["num_predict"] = 700
    raise RuntimeError("Model returned an empty response")

explanation = explain_tool_results({"inventory": stock_result, "tip": tip_result})
assert explanation.strip()
print(explanation[:800])
print("PASS: Qwen3 summarised tool output (content needs human accuracy review)")

## Cell 7 — Guard an unauthorized action

In [ ]:
try:
    dispatch_tool("issue_refund", {"amount": 500})
except PermissionError:
    print("PASS: Disallowed refund tool rejected")
else:
    raise AssertionError("Unlisted tool unexpectedly executed")
assert "issue_refund" not in [item["tool"] for item in TRACE]
print("PASS: Tool allowlist and trace integrity")

## Cell 8 — Save local source-practice evidence

In [ ]:
report = {
    "section": 2, "model": MODEL,
    "scope": "Deterministic Python tool dispatch plus local LLM explanation",
    "not_claimed": "Autonomous tool choice by Strands, AWS tools, or production safeguards",
    "tool_calls": TRACE,
    "answer": explanation,
    "passes": 7
}
destination = Path("section2_local_tools_report.json")
destination.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
assert destination.exists()
print("SAVED:", destination.resolve())
print("PASS: Section 2 local tools lab")